# QFF 2026 · Quantum Optimization · PS-O3 — Resource-Constrained Item Selection (0/1 Knapsack)
**Team:** YOUR_TEAM_NAME &nbsp;|&nbsp; **Problem statement:** PS-O3 &nbsp;|&nbsp; Run top to bottom, no edits needed (≈ 10-20 min).

Pipeline: exact classical optimum → QUBO (5 item + 4 slack qubits, P = 5) → Ising → p = 1 QAOA (5 seeded starts)
→ Processors A / B / D, ideal + noisy → resource metrics → Bell state on D → objective landscape on D (noisy).

In [ ]:
# ---------------- configuration ----------------
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()                      # notebook lives in the project root
sys.path.insert(0, str(ROOT))
from src import knapsack as K, qaoa_numpy as Qn, circuits as C, bell, landscape as L, results_io as R
from src import design_d as DD
from src.processors import load_processor, save_processor, validate_processor_d

TEAM_NAME = "YOUR_TEAM_NAME"
RECOMPUTE = False        # True = recompute the 20x20 noisy landscape (slow). False = reload saved grid if present.
RUN_D_SEARCH = False     # True = (re)run the Processor-D topology search (section 5). Does NOT overwrite D.
CHOSEN_D = None          # e.g. "circulant_n9_o1-2": build + SAVE processors/processor_D.json from the search. Then set back to None.
SEEDS = (0, 1, 2, 3, 4)  # official seeds
GAMMA_MAX, N_GRID = 1.0, 20

TAB, FIG = ROOT/"results"/"tables", ROOT/"results"/"figures"
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

import qiskit, qiskit_aer
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__, "| numpy", np.__version__)
print("official settings: basis", C.BASIS_GATES, "| opt level", C.OPT_LEVEL, "| seeds (transpiler, simulator)", C.SEED, "| shots", C.SHOTS)

## 1. Exact classical solution (the answer key)
Exhaustive search over all 2⁵ = 32 item subsets.

In [ ]:
table, best_classical = K.exhaustive_search()
df_items = pd.DataFrame(table)
print(f"Feasible subsets: {K.N_FEASIBLE_SUBSETS} of 32  ->  uniform-random P_feasible = {K.N_FEASIBLE_SUBSETS/32:.3f}")
print(f"Exact optimum V* = {K.V_STAR}  items {best_classical['items']}  weight {best_classical['weight']} (capacity {K.CAPACITY})")
df_items[df_items.feasible].sort_values("value", ascending=False).head(8)

## 2. Mathematical formulation and QUBO
Variables: $x_1..x_5$ (items, qubits 0-4) and slack bits $s_0..s_3$ (qubits 5-8), $s = s_0+2s_1+4s_2+8s_3 \in [0,15]$.

$$\min_{x,s}\; -\sum_i v_i x_i \;+\; P\Big(\sum_i w_i x_i + s_0 + 2s_1 + 4s_2 + 8s_3 - 15\Big)^2,\qquad P = 5$$

With $c = (5,3,4,2,6,1,2,4,8)$ and $y_i^2 = y_i$:  
$Q_{ii} = -v_i + P(c_i^2 - 30 c_i)$, $\;Q_{ij} = 2P\,c_i c_j\ (i<j)$, constant $= 225P$ (does not change the minimiser).

In [ ]:
Q, const = K.build_qubo()
names = [f"x{i+1}" for i in range(5)] + [f"s{i}" for i in range(4)]
display(pd.DataFrame(Q, index=names, columns=names).astype(int))
print("constant:", const)

# verification 1: matrix == formula for all 512 bitstrings
assert all(abs(K.bits(b,9)@Q@K.bits(b,9) + const - K.qubo_cost_direct(K.bits(b,9))) < 1e-9 for b in range(512))
# verification 2: QUBO minimum == exact classical optimum
m = K.QUBO_MIN_INDEX
print(f"QUBO minimum: cost {K.COST[m]:.0f} at items {K.items_from_index(m & K.ITEM_MASK)}, slack = {m >> 5}  (14 + 1 = 15, constraint tight)")
assert (m & K.ITEM_MASK) == K.OPT_INDEX and K.COST[m] == -K.V_STAR

## 3. Ising Hamiltonian and Hamiltonian scaling
$y_i = (1 - Z_i)/2$ gives $H = \text{offset} + \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$. All coefficients are divided by the largest magnitude (`scale`) so that γ is O(1). The scaling only re-parametrises γ.

In [ ]:
h, J, offset = K.to_ising(Q, const)
scale = K.ising_scale(h, J)
assert np.allclose(K.ising_diag(h, J, offset), K.COST)
diag_scaled = (K.COST - offset) / scale        # (constant dropped: global phase only)
print(f"{len(J)} ZZ terms (dense), {int((h!=0).sum())} Z terms, offset {offset}, scale (largest |coefficient|) = {scale}")
pd.DataFrame({"qubit": names, "h_i": h, "h_i/scale": h/scale}).round(3)

## 4. QAOA (p = 1): seeded optimisation
5 starts, `numpy.random.default_rng(seed)` with seeds 0-4, γ and β ~ U[0, π]. Optimiser **COBYLA** (scipy), max 200 iterations, cost = **exact statevector** expectation of the (scaled) QUBO Hamiltonian.

In [ ]:
runs, best = Qn.optimise_seeds(diag_scaled, seeds=SEEDS, maxiter=Qn.MAXITER)
df_runs = pd.DataFrame(runs).round(4)
display(df_runs)
print(f"Optimiser: {Qn.OPTIMIZER_NAME}, maxiter={Qn.MAXITER}, exact statevector cost")
print(f"BEST: seed {best['seed']}  gamma={best['gamma']:.4f}  beta={best['beta']:.4f}  <C>={best['expectation']:.4f}")
print(f"Consistency over seeds (ideal, exact): P_feasible = {df_runs.p_feasible_exact.mean():.3f} +/- {df_runs.p_feasible_exact.std():.3f} | "
      f"P_opt = {df_runs.p_opt_exact.mean():.4f} +/- {df_runs.p_opt_exact.std():.4f}")
df_runs.to_csv(TAB/"qaoa_seeds.csv", index=False)
gamma_opt, beta_opt = best["gamma"], best["beta"]

## 5. The QAOA circuit

In [ ]:
qc = C.build_qaoa_circuit(gamma_opt, beta_opt, h, J, scale)
print("logical qubits:", qc.num_qubits, "| gates:", dict(qc.count_ops()))
try:
    fig = qc.draw("mpl", fold=24, scale=0.6); fig.savefig(FIG/"qaoa_circuit.png", dpi=120, bbox_inches="tight"); display(fig)
except Exception as e:
    print("circuit drawing skipped:", e); print(qc.draw("text", fold=120))

## 6. Processors A, B (fixed by the kit) and D (our design)
D is **frozen** in `processors/processor_D.json`. The next cell can re-run the topology search (`RUN_D_SEARCH = True`): every legal candidate
(connected, degree ≤ 4, 9-12 qubits) is transpiled with the official settings and ranked by CX count / depth / SWAPs against Processor B.

In [ ]:
if RUN_D_SEARCH:
    df_search = DD.search_topologies(qc)
    df_search.to_csv(TAB/"d_search.csv", index=False)
    display(df_search.head(12))
    df_q = DD.quality_check(df_search, qc, top_k=5)
    df_q.to_csv(TAB/"d_search_quality.csv", index=False)
    display(df_q)
    if CHOSEN_D:
        procD = DD.build_chosen(CHOSEN_D, df_search, TEAM_NAME)
        save_processor(procD, ROOT/"processors"/"processor_D.json")
        print("Saved processors/processor_D.json for", CHOSEN_D, "-> now FREEZE it (set CHOSEN_D = None, commit).")
else:
    print("D search skipped (RUN_D_SEARCH = False). Using the frozen processors/processor_D.json.")

In [ ]:
PROC_A = load_processor(ROOT/"processors"/"processor_A.json")
PROC_B = load_processor(ROOT/"processors"/"processor_B.json")
PROC_D = load_processor(ROOT/"processors"/"processor_D.json")
ok, msgs = validate_processor_d(PROC_D, required_qubits=K.N_QUBITS)
print("Processor D validation:", *msgs, sep="\n  ")
assert ok, "Processor D failed validation -> would score 0 for Objective 2"
print("D rationale:", PROC_D.get("design_rationale", ""))

import networkx as nx
from src.processors import to_graph
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, P in zip(axes, (PROC_A, PROC_B, PROC_D)):
    g = to_graph(P)
    nx.draw(g, nx.circular_layout(g) if P is PROC_D else nx.kamada_kawai_layout(g), ax=ax, with_labels=True,
            node_color="#cfe3ff", edge_color="#555", node_size=520)
    ax.set_title(f"{P['name']}: {P['num_qubits']} qubits, {g.number_of_edges()} edges, max degree {max(dict(g.degree()).values())}")
fig.savefig(FIG/"processors_graphs.png", dpi=150, bbox_inches="tight")

## 7. Resource metrics (kit Section 9)
Measured on the bound QAOA circuit with final measurements. Processor A has 5 qubits, the circuit needs 9 → **not executable** (capacity rule).

In [ ]:
PROCS = {"A": PROC_A, "B": PROC_B, "D": PROC_D}
METRICS = {}
for lab, P in PROCS.items():
    if C.is_executable(qc, P):
        METRICS[lab] = C.circuit_metrics(qc, P)
df_res = pd.DataFrame(METRICS).T
display(df_res)
print(f"Processor A: not executable: needs {qc.num_qubits} qubits, has {PROC_A['num_qubits']}. The 4 slack qubits make the constraint exact; "
      "removing them to fit would change the locked problem.")
df_res.to_csv(TAB/"resources.csv")

## 8. Sampling: 4096 shots on every processor, ideal and noisy
Feasibility and value use the **5 item bits only** (slack ignored). Noise model: 1q depolarizing 0.001, 2q depolarizing 0.005, readout flip 0.01 (D uses the rates in its JSON).

In [ ]:
kit_rows, detail_rows = [], []
for lab, P in PROCS.items():
    for cond, noisy in (("ideal", False), ("noisy", True)):
        if lab not in METRICS:
            kit_rows.append(R.not_executable_row(lab, cond, P["num_qubits"]))
            detail_rows.append(dict(processor=lab, condition=cond, executable=False,
                                    note=f"not executable: needs {qc.num_qubits} qubits, has {P['num_qubits']}"))
            continue
        counts, tqc = C.sample(qc, P, noisy, C.SHOTS)
        ev = K.evaluate_counts(counts)
        (ROOT/"results"/f"counts_{lab}_{cond}.json").write_text(json.dumps(counts))
        kit_rows.append(R.result_row(lab, cond, METRICS[lab], ev, C.SHOTS))
        detail_rows.append(dict(processor=lab, condition=cond, executable=True, **ev, **METRICS[lab]))

rb = K.random_baseline()
df_detail = pd.DataFrame(detail_rows)
cols = ["processor","condition","executable","p_feasible","p_opt","best_feasible_value","best_feasible_items",
        "best_feasible_weight","pct_optimal","most_sampled_feasible_items","mean_qubo_cost"]
display(df_detail[[c for c in cols if c in df_detail.columns]].round(4))
print(f"Uniform-random baseline: P_feasible = {rb['p_feasible']:.3f}, P_opt = {rb['p_opt']:.4f}; exact optimum V* = {K.V_STAR}")
print("NOTE: with 4096 shots a random sampler also hits the optimal subset (P=%.3f) -> %% of optimal VALUE saturates at 100; "
      "compare P_feasible and P_opt against the random baseline instead." % rb["p_best_feasible_is_optimal_in_4096_shots"])
R.write_results_csv(kit_rows, TAB/"results.csv")
R.write_detail_csv(detail_rows, TAB/"results_detail.csv")

## 9. Objective 1: Bell state |Φ⁺⟩ on Processor D
F = (1 + ⟨XX⟩ − ⟨YY⟩ + ⟨ZZ⟩)/4. One adjacent pair and the farthest pair of D, ideal and noisy (12 circuits).

In [ ]:
df_bell = pd.DataFrame(bell.run_all(PROC_D))
df_bell = df_bell[["pair_type","pair","distance","condition","cx_count","XX","XX_se","YY","YY_se","ZZ","ZZ_se","F","F_se"]]
display(df_bell.round(4))
df_bell.to_csv(TAB/"bell.csv", index=False)

## 10. Objective landscape on Processor D (official noise)
Grid 20 × 20, β ∈ [0, π], γ ∈ [0, 1.0] (scaled Hamiltonian, divided by its largest coefficient **120**), 1024 shots/point, value = sampled mean of the true QUBO cost.
The ideal exact landscape is shown for comparison. Saved in `results/landscape_D_noisy.npz` (`RECOMPUTE = False` reloads it).

In [ ]:
gammas, betas = np.linspace(0, GAMMA_MAX, N_GRID), np.linspace(0, np.pi, N_GRID)
t0 = time.time()
Z, gammas, betas = L.load_or_compute(ROOT/"results"/"landscape_D_noisy.npz", RECOMPUTE,
                                     h=h, J=J, scale=scale, proc=PROC_D, gammas=gammas, betas=betas)
print(f"landscape ready ({time.time()-t0:.0f}s)")
Zi = Qn.landscape_exact(diag_scaled, gammas, betas, observable=K.COST)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
L.plot_landscape(Z, gammas, betas, gamma_opt, beta_opt, "Processor D, noisy (1024 shots/point)", ax=axes[0])
L.plot_landscape(Zi, gammas, betas, gamma_opt, beta_opt, "Ideal, exact statevector", ax=axes[1])
fig.savefig(FIG/"landscape_D.png", dpi=150, bbox_inches="tight")
i, j = np.unravel_index(Z.argmin(), Z.shape)
print(f"noisy grid minimum {Z.min():.2f} at gamma={gammas[j]:.3f}, beta={betas[i]:.3f}; ideal grid minimum {Zi.min():.2f}; random-guess cost {K.COST.mean():.1f}")

## 11. Summary (paste into README) and unofficial estimate of the rubric's objective formula (Processor B, noisy)

In [ ]:
rowB = df_detail[(df_detail.processor == "B") & (df_detail.condition == "noisy")].iloc[0]
s = lambda x, lo, hi: float(np.clip((x - lo) / (hi - lo), 0, 1))
formula = 3 * rowB.pct_optimal / 100 + 3 * s(rowB.p_feasible, 0.875, 0.909)
print(f"B noisy: % optimal = {rowB.pct_optimal:.1f}, P_feasible = {rowB.p_feasible:.4f}, P_opt = {rowB.p_opt:.4f}  ->  objective-formula points ~ {formula:.1f} / 6 (unofficial; the judges' script decides)")
summary = df_detail[df_detail.executable == True][["processor","condition","p_feasible","p_opt","best_feasible_value","best_feasible_weight","pct_optimal","cx","depth","swaps"]].round(4)
summary.to_csv(TAB/"summary_for_readme.csv", index=False)
try:
    print(summary.to_markdown(index=False))
except ImportError:
    print(summary.to_string(index=False))
print("\nDone. Files written to results/tables and results/figures.")